# ПЗ 6 — Классификация кадров через ResNet

Прогоняем кадры через предобученный ResNet50 (ImageNet), получаем топ-5 классов для каждого кадра.

In [ ]:
!pip install torch torchvision Pillow tqdm -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
VIDEO_DIR   = f'{BASE_DRIVE}/видио'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

for d in [VIDEO_DIR, FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'кадров: {len(frames)}')

In [ ]:
import json
import torch
import urllib.request
from torchvision import models, transforms

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json',
    '/content/imagenet_labels.json'
)
with open('/content/imagenet_labels.json') as f:
    LABELS = json.load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(device).eval()
print(f'resnet50 готов, устройство: {device}')

In [ ]:
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

rows = []

with torch.no_grad():
    for fname in tqdm(frames, desc='resnet'):
        img = Image.open(f'{FRAMES_DIR}/{fname}').convert('RGB')
        inp = transform(img).unsqueeze(0).to(device)
        probs = torch.softmax(model(inp), dim=1)[0]
        top5 = torch.topk(probs, 5)
        for rank, (idx, prob) in enumerate(zip(top5.indices.tolist(), top5.values.tolist()), 1):
            rows.append({'frame': fname, 'rank': rank, 'class': LABELS[idx], 'prob': round(prob, 4)})

df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/resnet_classifications.csv', index=False)
print(f'обработано кадров: {len(frames)}')
df[df['rank'] == 1].head(10)
